# Socratic Math Tutor: Kaggle runner (run as **Save Version → Save & Run All (Commit)**)

Run this notebook **3 times** (3 committed versions). Each part stays well under Kaggle's 12 h limit, and each part resumes
from the previous part's saved output:

| Part | `STAGES` | `RESUME_DIR` | ≈ GPU time |
|---|---|---|---|
| 1 | `data train ablations` | `""` | 5–6 h |
| 2 | `dpo eval judge` | output of Part 1 | 3–4 h |
| 3 | `sim gsm8k report` | output of Part 2 | 2–3 h |

Settings (right panel): Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = Files (optional).

In [ ]:
# ======== EDIT THESE TWO LINES FOR EACH PART ========
STAGES = "data train ablations"      # Part 2: "dpo eval judge"   Part 3: "sim gsm8k report"
RESUME_DIR = ""                      # Part 2/3: "/kaggle/input/<previous-version-output>/socratic-tutor-app"
REPO_URL = "https://github.com/<YOUR_USERNAME>/socratic-tutor-app.git"
# =====================================================

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("socratic-tutor-app"):
    if os.path.exists("/kaggle/input/socratic-tutor-app-code"):          # option B: code uploaded as a Kaggle Dataset
        subprocess.run("cp -r /kaggle/input/socratic-tutor-app-code /kaggle/working/socratic-tutor-app", shell=True, check=True)
    else:                                                                 # option A: public GitHub repo
        subprocess.run(f"git clone {REPO_URL}", shell=True, check=True)
os.chdir("/kaggle/working/socratic-tutor-app")
# Optional: Add-ons -> Secrets -> HF_TOKEN (faster, rate-limit-free model downloads)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print(os.getcwd())

In [ ]:
# Resume: bring back data/ and outputs/ from the previous part
if RESUME_DIR:
    assert os.path.exists(RESUME_DIR), f"attach the previous version's output as input first: {RESUME_DIR}"
    !cp -rn {RESUME_DIR}/data {RESUME_DIR}/outputs . 2>/dev/null; true
!ls data outputs 2>/dev/null | head -30

In [ ]:
!pip install -q unsloth
!pip install -q -r requirements-train.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m pytest -q tests   # sanity checks (CPU, a few seconds)

In [ ]:
!bash scripts/run_pipeline.sh {STAGES} 2>&1 | tee -a pipeline.log

In [ ]:
# Show what was produced in this part
!cat outputs/*/train_metrics.json 2>/dev/null | grep -E '"run"|n_train|train_hours|peak_vram|final_eval_loss' ; true
!tail -5 outputs_sft_raw.log outputs_sft_full_loss.log 2>/dev/null; true
if os.path.exists("results/RESULTS.md"):
    from IPython.display import Markdown, display
    display(Markdown(open("results/RESULTS.md").read()))
    display(Markdown(open("results/resume_bullets.md").read()))

In [ ]:
# Keep the saved output small: drop intermediate checkpoints (best adapter is already in outputs/*/adapter)
!rm -rf outputs/*/checkpoints
# One small zip with everything you need locally (adapters + metrics + eval outputs + results, no merged 3 GB model)
!zip -qr /kaggle/working/socratic_results.zip results outputs/*/adapter outputs/*/train_metrics.json outputs/eval data/data_stats.json data/eval_items.jsonl 2>/dev/null; true
!du -sh /kaggle/working/* | sort -h | tail